# Standard conditional diffusion — final field only (Colab)

Trains **standard** DDPM: predict noise when denoising **only the terminal field** \(u(\cdot, T)\) given the **boundary** channel. Intermediate PDE times are **not** used in the loss.

**Evaluation** uses the **same** final-field MSE on `test.pt` as the trajectory-aligned notebook.

**GPU:** A100 recommended. **Repo** on Google Drive — set `REPO` below.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Point to the repo on Drive

Edit `REPO` to the folder containing `src/`, `configs/`, `outputs/`.

In [ ]:
import os
import sys

# --- EDIT THIS PATH ---
REPO = "/content/drive/MyDrive/path/to/heat_operator_recovery"
# ----------------------

os.chdir(REPO)
sys.path.insert(0, REPO)
print("CWD:", os.getcwd())

## 3. Dependencies

In [ ]:
!pip install -q torch numpy matplotlib tqdm pyyaml

## 4. GPU check

In [ ]:
import torch
assert torch.cuda.is_available(), "Enable GPU: Runtime → Change runtime type → GPU"
print(torch.cuda.get_device_name(0))

## 5. Train (standard — boundary → final field only)

Checkpoint: `outputs/checkpoints/model_standard_final.pt`

In [ ]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "src.training.train_standard",
        "--config",
        "configs/default.yaml",
        "--output",
        "model_standard_final.pt",
    ],
    cwd=REPO,
    check=True,
)

## 6. Evaluate — **final field only** (test split)

In [ ]:
from pathlib import Path

import torch

from src.models.diffusion import GaussianDiffusion
from src.models.unet import SmallUNet
from src.training.eval_metrics import dataset_mean_final_mse
from src.utils.config import load_config

root = Path(REPO)
cfg = load_config(root / "configs" / "default.yaml")
device = torch.device("cuda")

ckpt_path = root / "outputs" / "checkpoints" / "model_standard_final.pt"
ckpt = torch.load(ckpt_path, map_location=device)
num_diff = int(ckpt["diffusion"]["num_timesteps"])

diffusion = GaussianDiffusion(num_timesteps=num_diff).to(device)
model = SmallUNet(in_ch=2, base=32, time_dim=128).to(device)
model.load_state_dict(ckpt["model"])

test_pt = root / "outputs" / "data" / "test.pt"
bs = int(cfg.get("batch_size", 16))
mse = dataset_mean_final_mse(model, diffusion, test_pt, batch_size=bs, device=device)
print("Standard (final-only) model — test final-state MSE:", mse)